In [ ]:
%pip install opencv-python --quiet
%pip install seaborn --quiet
%pip install torchsummary --quiet
%pip install imgaug --quiet

In [ ]:
import os
import cv2
import torch
import pathlib
import random
import multiprocessing
import numpy as np
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
import torch.nn.functional as F
import lightning as L

from lightning.pytorch.loggers import CSVLogger

from torchsummary import summary
from imgaug import augmenters as iaa

from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_image
from torchvision.transforms import v2

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

In [ ]:
PATH = pathlib.Path("./webots_lanes")
FILENAME = "dataset.csv"

In [ ]:
angles_df = pd.read_csv(PATH / FILENAME)

angles_df.sample(10)

In [ ]:
num_bins = 50
samples_per_bin = 1500
hist, bins = np.histogram(angles_df['steering_angle'], num_bins)
center = (bins[:-1]+ bins[1:]) * 0.5

sb.histplot(angles_df['steering_angle'], bins=num_bins)

In [ ]:
remove_list = []

for j in range(num_bins):
    list_ = []
    for i in range(len(angles_df['steering_angle'])):
        if angles_df['steering_angle'][i] >= bins[j] and angles_df['steering_angle'][i] <= bins[j+1]:
            list_.append(i)
    list_ = shuffle(list_)
    list_ = list_[samples_per_bin:]
    remove_list.extend(list_)

print("removed: ", len(remove_list))
angles_df.drop(angles_df.index[remove_list], inplace=True)
print("remaining:", len(angles_df))

sb.histplot(angles_df['steering_angle'], bins=num_bins)

In [ ]:
train, val_test = train_test_split(angles_df, train_size=0.7, random_state=43, shuffle=True)
val, test = train_test_split(val_test, train_size=0.5, random_state=17, shuffle=True)

len(train), len(val), len(test)

In [ ]:
class ImageDataset(Dataset):
    
    def __init__(self, annotations_df, img_dir, transform=None, img_transforms=None) -> None:
        self.img_labels = annotations_df
        self.img_dir = img_dir
        self.transform = transform
        self.img_transforms=img_transforms
        
    def __len__(self):
        return len(self.img_labels)
    
    def __getitem__(self, index):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[index, 0])
        image = cv2.imread(img_path)
        label = torch.tensor(self.img_labels.iloc[index, 1], dtype=torch.float32)
        if self.transform:
            image, label = self.transform(image, label)
        if self.img_transforms:
            image = self.img_transforms(image)
        return image, label

In [ ]:
class RandomZoomTransform(torch.nn.Module):
    
    def __init__(self, threshold=0.5, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.threshold = threshold
    
    def forward(self, image, label):
        if random.random() > self.threshold:
            zoom = iaa.Affine(scale=(1, 1.3))
            image = zoom.augment_image(image)
        return image, label


class RandomPanTransform(torch.nn.Module):
    
    def __init__(self, threshold=0.5, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.threshold = threshold
    
    def forward(self, image, label):
        if random.random() > self.threshold:
            pan = iaa.Affine(translate_percent={"x": (-0.1,0.1), "y": (-0.1, 0.1)})
            image = pan.augment_image(image)
        return image, label


class RandomBrightnessTransform(torch.nn.Module):
    
    def __init__(self, threshold=0.5, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.threshold = threshold
    
    def forward(self, image, label):
        if random.random() > self.threshold:
            brightness = iaa.Multiply((0.2, 1.2))
            image = brightness.augment_image(image)
        return image, label


class FlipTransform(torch.nn.Module):
    
    def __init__(self, threshold=0.5, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.threshold = threshold
    
    def forward(self, image, label):
        if random.random() > self.threshold:
            image = cv2.flip(image,1)
            label = -label
        return image, label

In [ ]:
aug_transforms = v2.Compose([
    RandomZoomTransform(),
    RandomPanTransform(),
    RandomBrightnessTransform(),
    FlipTransform()
])

img_transforms = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])


path = str(PATH / "Train")


train_dataset = ImageDataset(train, path, transform=aug_transforms, img_transforms=img_transforms)
val_dataset = ImageDataset(val, path, transform=aug_transforms, img_transforms=img_transforms)
test_dataset = ImageDataset(test, path, transform=aug_transforms, img_transforms=img_transforms)


train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=multiprocessing.cpu_count())
val_dataloader = DataLoader(val_dataset, batch_size=32, num_workers=multiprocessing.cpu_count())
test_dataloader = DataLoader(test_dataset, batch_size=32, num_workers=multiprocessing.cpu_count())

In [ ]:
class Conv2dModule(torch.nn.Module):
    
    def __init__(self, input, output, kernel, stride=1, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.conv = torch.nn.Conv2d(input, output, kernel, stride=stride)
        self.elu = torch.nn.ELU()

    def forward(self, image):
        x = self.conv(image)
        x = self.elu(x)
        return x


class LinearModule(torch.nn.Module):
    
    def __init__(self, input, output, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.linear = torch.nn.Linear(input, output)
        self.elu = torch.nn.ELU()

    def forward(self, image):
        x = self.linear(image)
        x = self.elu(x)
        return x


class NvidiaModel(L.LightningModule):

    def __init__(self, *args, **kwargs) -> None:
        super(NvidiaModel, self).__init__(*args, **kwargs)
        self.conv1 = Conv2dModule(3, 24, 5, stride=(2,2))
        self.conv2 = Conv2dModule(24, 36, 5, stride=(2,2))
        self.conv3 = Conv2dModule(36, 48, 5, stride=(2,2))
        self.conv4 = Conv2dModule(48, 64, 3)
        self.conv5 = Conv2dModule(64, 64, 3)

        self.flatten = torch.nn.Flatten()

        self.linear1 = LinearModule(1152, 100)
        self.linear2 = LinearModule(100, 50)
        self.linear3 = LinearModule(50, 10)
        self.linear4 = LinearModule(10, 1)
        
    
    def training_step(self, batch, batch_idx):
        x, y = batch
        
        y_hat = self(x)

        loss = torch.sqrt(F.mse_loss(y_hat, y.unsqueeze(1)))
        self.log("train_loss", loss, on_epoch=True)
        return loss
    
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = torch.sqrt(F.mse_loss(y_hat, y.unsqueeze(1)))
        self.log("val_loss", loss, on_epoch=True)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = torch.sqrt(F.mse_loss(y_hat, y.unsqueeze(1)))
        return loss
    
    def predict_step(self, batch, batch_idx):
        x, _ = batch
        y_hat = self(x)
        return y_hat

    def configure_optimizers(self):
        optim = torch.optim.Adam(self.parameters(), lr=1e-3)
        return optim


    def forward(self, x):
        for i in range(1, 6):
            x = getattr(self, f'conv{i}')(x)

        x = self.flatten(x)

        for i in range(1, 5):
            x = getattr(self, f'linear{i}')(x)

        return x

In [ ]:
nvidiaModel = NvidiaModel().to('cuda')

summary(nvidiaModel, input_size=(3,66,200))

In [ ]:
logger = CSVLogger('./csv_logs', name='nvidia_model_logs', version='0.1.0')
torch.set_float32_matmul_precision('medium')
trainer = L.Trainer(max_epochs=20, accelerator='gpu', precision='16', logger=logger)
trainer.fit(model=nvidiaModel, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

In [ ]:
test = test.sample(1)
image = cv2.imread(str(PATH / "Train" / test.iloc[0,0]))

tensor = img_transforms(image)
nvidiaModel.eval()
angle = nvidiaModel(tensor.unsqueeze(0))

plt.imshow(image)
plt.title(f"steering angle: {angle[0][0]}, {test.iloc[0,1]}")

In [ ]:
nvidiaModel.eval()

input = torch.randn(1, 3, 200, 66)
torch.onnx.export(nvidiaModel, input, '', verbose=True)